## Изначальный скрипт

In [1]:
import json
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report
import nltk
from nltk.corpus import stopwords
import re

# Download Russian stopwords
nltk.download('stopwords')
russian_stopwords = set(stopwords.words('russian'))


# Load the JSONL file
def load_jsonl(file_path):
   data = []
   with open(file_path, 'r', encoding='utf-8') as f:
       for line in f:
           data.append(json.loads(line))
   return pd.DataFrame(data)

# Preprocess text
def preprocess_text(text):
   # Convert to lowercase
   text = text.lower()
   # Remove punctuation
   text = re.sub(r'[^\w\s]', '', text)
   # Remove stopwords
   words = text.split()
   words = [word for word in words if word not in russian_stopwords]
   return ' '.join(words)

# Load the data
df = load_jsonl('kinopoisk.jsonl')

# Preprocess the content
df['processed_content'] = df['content'].apply(preprocess_text)

# Split the data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
   df['processed_content'], df['grade3'], test_size=0.2, random_state=42
)

# Create bag of words representation
vectorizer = CountVectorizer()
X_train_bow = vectorizer.fit_transform(X_train)
X_test_bow = vectorizer.transform(X_test)

# Train the classifier
clf = MultinomialNB()
clf.fit(X_train_bow, y_train)

# Make predictions
y_pred = clf.predict(X_test_bow)

# Evaluate the model
print(classification_report(y_test, y_pred))


# Function to classify new reviews
def classify_review(review_text):
   processed_text = preprocess_text(review_text)
   bow_representation = vectorizer.transform([processed_text])
   prediction = clf.predict(bow_representation)
   return prediction[0]

# Example usage
#example_review = "Ваш текст рецензии на русском языке"
#print(f"Классификация: {classify_review(example_review)}")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


              precision    recall  f1-score   support

         Bad       0.63      0.65      0.64       979
        Good       0.84      0.97      0.90      5412
     Neutral       0.41      0.02      0.03       928

    accuracy                           0.81      7319
   macro avg       0.62      0.55      0.52      7319
weighted avg       0.76      0.81      0.76      7319



In [2]:
import mlflow
mlflow.set_tracking_uri("http://localhost:5000")  # Замените на ваш адрес сервера MLflow
mlflow.set_experiment("MovieReviewSentiment")

<Experiment: artifact_location='mlflow-artifacts:/127249901640666205', creation_time=1730276033594, experiment_id='127249901640666205', last_update_time=1730276033594, lifecycle_stage='active', name='MovieReviewSentiment', tags={}>

## MultinomialNB

In [3]:
import pandas as pd
import mlflow
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Start an MLflow run
with mlflow.start_run():
    # Download Russian stopwords
    nltk.download('stopwords')
    russian_stopwords = set(stopwords.words('russian'))
    
    
    # Load the JSONL file
    def load_jsonl(file_path):
       data = []
       with open(file_path, 'r', encoding='utf-8') as f:
           for line in f:
               data.append(json.loads(line))
       return pd.DataFrame(data)
    
    # Preprocess text
    def preprocess_text(text):
       # Convert to lowercase
       text = text.lower()
       # Remove punctuation
       text = re.sub(r'[^\w\s]', '', text)
       # Remove stopwords
       words = text.split()
       words = [word for word in words if word not in russian_stopwords]
       return ' '.join(words)
    
    # Load the data
    df = load_jsonl('kinopoisk.jsonl')
    
    # Preprocess the content
    df['processed_content'] = df['content'].apply(preprocess_text)

    # Split the data into train and test sets
    X_train, X_test, y_train, y_test = train_test_split(
       df['processed_content'], df['grade3'], test_size=0.2, random_state=42
    )
    
    # Create bag of words representation
    vectorizer = CountVectorizer()
    X_train_bow = vectorizer.fit_transform(X_train)
    X_test_bow = vectorizer.transform(X_test)
    
    # Train the classifier
    clf = MultinomialNB()
    clf.fit(X_train_bow, y_train)
    
    # Make predictions
    y_pred = clf.predict(X_test_bow)
    
    # Evaluate the model
    print(classification_report(y_test, y_pred))
    
    
    # Function to classify new reviews
    def classify_review(review_text):
       processed_text = preprocess_text(review_text)
       bow_representation = vectorizer.transform([processed_text])
       prediction = clf.predict(bow_representation)
       return prediction[0]

    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='macro')
    recall = recall_score(y_test, y_pred, average='macro')
    f1 = f1_score(y_test, y_pred, average='macro')
    
    mlflow.log_param("model_name", "MultinomialNB")  # Записываем имя модели
    mlflow.log_param("random_state", 42)

    # Log metrics
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)

    # Log the model
    mlflow.sklearn.log_model(clf, "model")

    # Print results
    print(f"Accuracy: {accuracy:.2f}")
    print(f"Precision: {precision:.2f}")
    print(f"Recall: {recall:.2f}")
    print(f"F1 Score: {f1:.2f}")

    # Get the run ID
    run_id = mlflow.active_run().info.run_id
    print(f"MLflow Run ID: {run_id}")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


              precision    recall  f1-score   support

         Bad       0.63      0.65      0.64       979
        Good       0.84      0.97      0.90      5412
     Neutral       0.41      0.02      0.03       928

    accuracy                           0.81      7319
   macro avg       0.62      0.55      0.52      7319
weighted avg       0.76      0.81      0.76      7319



2024/10/30 14:40:10 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2024/10/30 14:40:10 INFO mlflow.tracking._tracking_service.client: 🏃 View run nervous-hawk-396 at: http://localhost:5000/#/experiments/127249901640666205/runs/b4920bf80ad9457bbbe195a9dc5c5879.
2024/10/30 14:40:10 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://localhost:5000/#/experiments/127249901640666205.


Accuracy: 0.81
Precision: 0.62
Recall: 0.55
F1 Score: 0.52
MLflow Run ID: b4920bf80ad9457bbbe195a9dc5c5879


## LogisticRegression

In [5]:
import mlflow
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer 
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, f1_score
import nltk
from nltk.corpus import stopwords
import re

with mlflow.start_run():
    # Download Russian stopwords
    nltk.download('stopwords')
    russian_stopwords = set(stopwords.words('russian'))
    
    
    # Load the JSONL file
    def load_jsonl(file_path):
       data = []
       with open(file_path, 'r', encoding='utf-8') as f:
           for line in f:
               data.append(json.loads(line))
       return pd.DataFrame(data)
    
    # Preprocess text
    def preprocess_text(text):
       # Convert to lowercase
       text = text.lower()
       # Remove punctuation
       text = re.sub(r'[^\w\s]', '', text)
       # Remove stopwords
       words = text.split()
       words = [word for word in words if word not in russian_stopwords]
       return ' '.join(words)
    
    # Load the data
    df = load_jsonl('kinopoisk.jsonl')
    
    # Preprocess the content
    df['grade3'], grade_labels = pd.factorize(df['grade3'])  
    df['processed_content'] = df['content'].apply(preprocess_text)

    # Split the data into train and test sets
    X_train, X_test, y_train, y_test = train_test_split(
       df['processed_content'], df['grade3'], test_size=0.2, random_state=42
    )
    # Create TF-IDF representation
    vectorizer = TfidfVectorizer(stop_words=list(russian_stopwords)) 
    X_train_bow = vectorizer.fit_transform(X_train)
    X_test_bow = vectorizer.transform(X_test)
   
    # Обучение модели
    clf = LogisticRegression(random_state=42)  # Используем LogisticRegression
    clf.fit(X_train_bow, y_train)

    # Предсказание
    y_pred = clf.predict(X_test_bow)

    # Оценка
    print(classification_report(y_test, y_pred))
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='macro')
    recall = recall_score(y_test, y_pred, average='macro')
    f1 = f1_score(y_test, y_pred, average='macro')
   
    # Log parameters
    mlflow.log_param("model", "LogisticRegression")
    mlflow.log_param("random_state", 42)

    # Log metrics
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)

    # Log the model
    mlflow.sklearn.log_model(clf, "LogisticRegression")

    # Print results
    print(f"Accuracy: {accuracy:.2f}")
    print(f"Precision: {precision:.2f}")
    print(f"Recall: {recall:.2f}")
    print(f"F1 Score: {f1:.2f}")

    # Get the run ID
    run_id = mlflow.active_run().info.run_id
    print(f"MLflow Run ID: {run_id}")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


              precision    recall  f1-score   support

           0       0.84      0.98      0.91      5412
           1       0.53      0.09      0.15       928
           2       0.75      0.63      0.69       979

    accuracy                           0.82      7319
   macro avg       0.71      0.57      0.58      7319
weighted avg       0.79      0.82      0.78      7319



2024/10/30 14:42:26 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2024/10/30 14:42:26 INFO mlflow.tracking._tracking_service.client: 🏃 View run incongruous-cub-171 at: http://localhost:5000/#/experiments/127249901640666205/runs/17d614de358744e49232a016b28156ba.
2024/10/30 14:42:26 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://localhost:5000/#/experiments/127249901640666205.


Accuracy: 0.82
Precision: 0.71
Recall: 0.57
F1 Score: 0.58
MLflow Run ID: 17d614de358744e49232a016b28156ba
